# Research metadata and provenance for Apache Atlas

## Simplified curation scenario

A research team studies situated vocabulary in Spanish-language podcasts and supporting literature. Its experimental workflow collects transcripts and reference documents, maintains a podcast register and a situated lexicon, and uses GitHub notebooks to produce frequency tables, indexes and term relationships. Curation must describe both the artifacts and the processes that create new versions of them.

This exercise starts with three SRT transcripts, two PDF documents, two Excel workbooks and two version-pinned GitHub notebooks. It inventories those inputs, profiles their structure, creates CSV extracts, and records the inputs, outputs, times, parameters and software of each transformation actually performed here. It then prepares Apache Atlas type definitions and entity payloads. The original audio, original transcription process and earlier experiments are not supplied; their provenance remains unknown. No speech recognition, vocabulary classification or external research notebook is executed.

The two accompanying workbooks extend the earlier classroom tables: **Data catalogue** records item, origin, format, creator, access and modification rights; **Metadata schema** defines reusable fields, rules, examples and Atlas mappings. The referenced GitHub folder contains analysis notebooks rather than these catalogue templates. Their analytical workflow informs this example, but their historical execution is not inferred from their source code.

## How to run

Keep this notebook beside `input/`, the two manifests and the two catalogue workbooks. Install Python 3.10 or later and the packages in `requirements.txt`, open this notebook in Jupyter, and run cells in order. The core workflow works offline once dependencies are installed. Optional Atlas submission is disabled. Every execution writes a new directory under `runs/`; supplied inputs and catalogue workbooks are never overwritten.

No emojis are used in the explanations. Document contents are research data, not executable instructions. File names, embedded PDF authors, podcast analysts and repository owners are not automatically treated as artifact creators. Access and editing rights need curator confirmation. The subject of a transcript or a vocabulary match must not be used to infer a speaker's identity.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import csv, hashlib, importlib.metadata, io, json, os, re, sys, uuid
from urllib.parse import quote
import openpyxl
from pypdf import PdfReader

# Set PROJECT_ROOT when Jupyter starts outside this extracted folder.
PROJECT_ROOT = Path.cwd().resolve()
INPUT = PROJECT_ROOT / "input"
NOTEBOOK = PROJECT_ROOT / "metadata_provenance_atlas.ipynb"
if not INPUT.is_dir() or not NOTEBOOK.is_file():
    raise FileNotFoundError("Set PROJECT_ROOT to the folder containing this notebook and input/.")

# Use a role or a researcher identifier when known. Unknown is not an inferred identity.
CURATOR_ID = "unknown"
PROJECT_ID = "queerdatagap"
CSV_ENCODING = "utf-8-sig"
CSV_DELIMITER = None  # None uses csv.Sniffer over a sample; set explicitly if needed.
HEADER_ROWS = {"QueerDataGap_Lexico_Situado.xlsx": 4, "lista_podcasts.xlsx": 3}
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "_" + uuid.uuid4().hex[:8]
RUN = PROJECT_ROOT / "runs" / RUN_ID
RUN.mkdir(parents=True, exist_ok=False)

def now():
    return datetime.now(timezone.utc).isoformat()

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def dump(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

def relative(path):
    return Path(path).relative_to(PROJECT_ROOT).as_posix()

def show(rows, limit=10):
    # Plain JSON is also readable outside an interactive notebook.
    print(json.dumps(rows[:limit], ensure_ascii=False, indent=2, default=str))

versions = {name: importlib.metadata.version(name) for name in ["openpyxl", "pypdf"]}
config = {"project_id": PROJECT_ID, "curator_id": CURATOR_ID,
          "csv_encoding": CSV_ENCODING, "csv_delimiter": CSV_DELIMITER,
          "header_rows": HEADER_ROWS, "python": sys.version, "packages": versions}
dump(RUN / "run_config.json", config)
print("Run:", RUN_ID)


Run: 20260915T172858_c692996a


## 1 Verify the input package and load curator annotations

SHA-256 checks detect changes to supplied files. The repository snapshot also records a Git commit and Git blob identifier. These identify the code version; they do not prove that the code generated an existing dataset. Filesystem modification time describes the local file and must not be relabelled as the creation date of the research data.

Edit `data_catalogue.xlsx` to document known creators, access, modification rights and source evidence. The notebook reads these annotations by relative path. It rejects duplicate paths. Automated measurements remain separate from curator statements, and every run preserves a copy of the annotations it used. Unknown rights do not mean that access is public.


In [2]:
manifest = json.loads((PROJECT_ROOT / "input_manifest.json").read_text(encoding="utf-8"))
repo_snapshot = json.loads((PROJECT_ROOT / "repository_snapshot.json").read_text(encoding="utf-8"))
source_lookup = {x["relative_path"]: x for x in manifest + repo_snapshot}
package_checks = []
for entry in manifest + repo_snapshot:
    path = PROJECT_ROOT / entry["relative_path"]
    package_checks.append({"path": entry["relative_path"], "exists": path.is_file(),
                           "hash_matches": path.is_file() and sha256(path) == entry["sha256"]})
if not all(x["hash_matches"] for x in package_checks):
    raise ValueError("An input is missing or changed. Review the manifests before proceeding.")

annotations = {}
catalogue_path = PROJECT_ROOT / "data_catalogue.xlsx"
if catalogue_path.exists():
    book = openpyxl.load_workbook(catalogue_path, read_only=True, data_only=False)
    sheet = book["Data catalogue"]
    iterator = sheet.iter_rows(values_only=True)
    headers = next(iterator)
    for values in iterator:
        row = dict(zip(headers, values))
        key = row.get("Relative path")
        if not key or not str(key).startswith("input/"):
            continue
        if key in annotations:
            raise ValueError(f"Duplicate catalogue path: {key}")
        annotations[key] = {k: row.get(k) for k in ["Source or origin", "Who creates it",
                           "Who can access it", "Who can modify it", "Evidence or notes"]}
    book.close()
dump(RUN / "curator_annotations.json", annotations)
show(package_checks)


[
  {
    "path": "input/transcripts/fvfRMweIiWw.srt",
    "exists": true,
    "hash_matches": true
  },
  {
    "path": "input/transcripts/h-CkQVuco1Q.srt",
    "exists": true,
    "hash_matches": true
  },
  {
    "path": "input/transcripts/rTYgX5Qtoqs.srt",
    "exists": true,
    "hash_matches": true
  },
  {
    "path": "input/documents/Glosario_TDSyG_WEB.pdf",
    "exists": true,
    "hash_matches": true
  },
  {
    "path": "input/documents/Resentir_lo_queer_en_America_Latina.pdf",
    "exists": true,
    "hash_matches": true
  },
  {
    "path": "input/tables/QueerDataGap_Lexico_Situado.xlsx",
    "exists": true,
    "hash_matches": true
  },
  {
    "path": "input/tables/lista_podcasts.xlsx",
    "exists": true,
    "hash_matches": true
  },
  {
    "path": "input/code/queerdatagap_clean_frequency_matrix_LGBTQ_vocabulary.ipynb",
    "exists": true,
    "hash_matches": true
  },
  {
    "path": "input/code/queerdatagraph_explorando_terminos_relaciones_SRT.ipynb",
    "exists": 

## 2 Define format-specific metadata extractors

The CSV reader preserves strings, checks row widths and reports empty cells. Its row count excludes the header. XLSX readers use the confirmed header rows (row 4 for the lexicon, row 3 for the podcast register) and exclude blank formatted rows. They retain source row numbers and formula expressions rather than relying on potentially stale cached results. CSV copies preserve those expressions as text; use a text editor or import columns as text when viewing them in spreadsheet software.

The SRT parser retains cue identifiers and start/end times in seconds, accepts comma or dot milliseconds, and reports malformed cues, reversed times and overlaps. Maximum cue end time is a transcript time extent, not a verified audio duration. PDF extraction reads embedded metadata and counts pages; it does not infer authorship from an embedded author string, perform OCR or reproduce document text. Notebook inspection parses JSON only and records cell counts, kernel information and source-code hashes without executing the inspected code.


In [3]:
def read_csv_table(path):
    text = Path(path).read_text(encoding=CSV_ENCODING)  # Fail visibly on an incorrect encoding.
    delimiter = CSV_DELIMITER
    if delimiter is None:
        try:
            delimiter = csv.Sniffer().sniff(text[:65536], delimiters=",;\t|").delimiter
        except csv.Error:
            delimiter = ","
    rows = list(csv.reader(io.StringIO(text), delimiter=delimiter))
    if not rows:
        return {"row_count": 0, "columns": [], "delimiter": delimiter, "empty_file": True}
    header, body = rows[0], rows[1:]
    return {"row_count": len(body), "column_count": len(header), "columns": header,
            "encoding": CSV_ENCODING, "delimiter": delimiter,
            "duplicate_headers": [k for k, n in Counter(header).items() if n > 1],
            "inconsistent_row_numbers": [i for i, r in enumerate(body, 2) if len(r) != len(header)],
            "empty_cells": sum(v == "" for r in body for v in r)}

def xlsx_tables(path):
    book = openpyxl.load_workbook(path, read_only=True, data_only=False)
    tables = []
    try:
        for sheet in book:
            rows = list(sheet.iter_rows(values_only=True))
            header_row = HEADER_ROWS.get(Path(path).name, 1)
            header_values = rows[header_row - 1] if len(rows) >= header_row else ()
            # Include columns used anywhere, even when their header is empty.
            width = max((i + 1 for row in rows for i, v in enumerate(row) if v is not None), default=0)
            headers = [str(header_values[i]) if i < len(header_values) and header_values[i] is not None
                       else f"unnamed_column_{i + 1}" for i in range(width)]
            seen = Counter()
            unique = []
            for h in headers:
                seen[h] += 1
                unique.append(h if seen[h] == 1 else f"{h}__{seen[h]}")
            body = [(i, list(row[:width]) + [None] * max(0, width - len(row)))
                    for i, row in enumerate(rows[header_row:], header_row + 1)
                    if any(v is not None for v in row)]
            tables.append({"sheet": sheet.title, "header_row": header_row, "columns": unique,
                           "original_headers": headers, "rows": body,
                           "formula_count": sum(isinstance(v, str) and v.startswith("=")
                                                for _, row in body for v in row),
                           "excluded_blank_rows": len(rows[header_row:]) - len(body)})
    finally:
        book.close()
    return tables

STAMP = r"(\d{2,}):([0-5]\d):([0-5]\d)[,.](\d{3})"
TIMING = re.compile(r"^" + STAMP + r"\s*-->\s*" + STAMP + r"(?:\s+.*)?$")

def parse_srt(path):
    text = Path(path).read_text(encoding="utf-8-sig").replace("\r\n", "\n").replace("\r", "\n")
    cues, problems = [], []
    for index, block in enumerate(re.split(r"\n\s*\n", text.strip()), 1):
        if not block.strip():
            continue
        lines = block.splitlines()
        line_index = next((i for i, line in enumerate(lines[:2]) if "-->" in line), None)
        match = TIMING.match(lines[line_index].strip()) if line_index is not None else None
        if not match:
            problems.append({"block": index, "issue": "Malformed or missing timestamp"})
            continue
        parts = list(map(int, match.groups()))
        start = parts[0]*3600 + parts[1]*60 + parts[2] + parts[3]/1000
        end = parts[4]*3600 + parts[5]*60 + parts[6] + parts[7]/1000
        if end < start:
            problems.append({"block": index, "issue": "End precedes start"})
            continue
        cue = {"source_block": index, "cue_id": lines[0].strip() if line_index == 1 else str(index),
               "start_seconds": start, "end_seconds": end,
               "text": "\n".join(lines[line_index + 1:])}
        if cues and start < cues[-1]["end_seconds"]:
            problems.append({"block": index, "issue": "Overlaps or precedes previous cue"})
        if not cue["text"].strip():
            problems.append({"block": index, "issue": "Empty cue text"})
        cues.append(cue)
    return cues, problems

def profile(path):
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return read_csv_table(path)
    if suffix == ".srt":
        cues, problems = parse_srt(path)
        return {"cue_count": len(cues), "max_cue_end_seconds": max((r["end_seconds"] for r in cues), default=None),
                "timestamp_issues": problems, "encoding": "utf-8-sig",
                "register_matches_unverified": REGISTER_ENTRIES.get(path.stem, [])}
    if suffix == ".xlsx":
        return {"sheets": [{k: v for k, v in t.items() if k != "rows"} | {"row_count": len(t["rows"])}
                           for t in xlsx_tables(path)]}
    if suffix == ".pdf":
        reader = PdfReader(path)
        if reader.is_encrypted and not reader.decrypt(""):
            raise ValueError("Encrypted PDF requires a password")
        return {"page_count": len(reader.pages), "encrypted": reader.is_encrypted,
                "embedded_metadata_unverified": {str(k): str(v) for k, v in (reader.metadata or {}).items()}}
    if suffix == ".ipynb":
        notebook = json.loads(path.read_text(encoding="utf-8"))
        cells = notebook.get("cells", [])
        code = "\n\n".join("".join(c.get("source", [])) for c in cells if c.get("cell_type") == "code")
        return {"cell_counts": dict(Counter(c.get("cell_type") for c in cells)),
                "kernel": notebook.get("metadata", {}).get("kernelspec", {}),
                "code_source_sha256": hashlib.sha256(code.encode()).hexdigest(), "executed_here": False}
    return {"profile_note": "Generic file metadata only"}


## 3 Inventory the collection

One entity represents one path and content version. Its qualified name includes the project, relative path and SHA-256. Repeating an export for unchanged content addresses the same Atlas dataset; changed bytes produce a new version entity. Renaming a file also changes this identifier. Paths are relative so the package can be moved without exposing a user's home directory.

The extraction status reports technical success only. It does not certify scholarly claims, access rights or metadata completeness. Each error is retained in the report rather than silently excluding the file. Exact transcript-file identifiers are matched against the supplied podcast register to recover listed episode titles, URLs and analysis assignments with source row numbers. These remain register statements, not independently verified web metadata. The assigned analyst is not treated as the transcript creator.


In [4]:
REGISTER_ENTRIES = {}
register_path = INPUT / "tables" / "lista_podcasts.xlsx"
if register_path.exists():
    for table in xlsx_tables(register_path):
        for row_number, values in table["rows"]:
            entry = dict(zip(table["columns"], values))
            identifier = entry.get("Nombre archivo transcripción")
            if identifier:
                key = Path(str(identifier).strip()).stem
                record = {"episode_title": entry.get("Episodio concreto"), "listed_url": entry.get("Enlace"),
                          "listed_platform": entry.get("Plataforma"), "listed_country": entry.get("País"),
                          "assigned_analyst": entry.get("Quién lo analiza"),
                          "evidence": relative(register_path) + f"#sheet={table['sheet']}&row={row_number}"}
                REGISTER_ENTRIES.setdefault(key, []).append(record)

assets, activities = [], []
FORMAT = {".srt": "SRT transcript", ".pdf": "PDF document", ".xlsx": "Excel workbook",
          ".csv": "CSV dataset", ".ipynb": "Jupyter notebook", ".py": "Python code", ".json": "JSON metadata"}

def asset(path, stage="input", origin=None):
    rel, digest = relative(path), sha256(path)
    source = source_lookup.get(rel, {})
    curated = annotations.get(rel, {})
    started = now()
    try:
        technical, status, error = profile(path), "ok", None
    except Exception as exc:
        technical, status, error = {}, "error", f"{type(exc).__name__}: {exc}"
    return {"qualified_name": f"{PROJECT_ID}://artifact/{quote(rel, safe='/')}@sha256:{digest}",
            "name": path.name, "relative_path": rel, "sha256": digest, "size_bytes": path.stat().st_size,
            "format": FORMAT.get(path.suffix.lower(), path.suffix.lstrip('.').upper()), "stage": stage,
            "source_origin": origin or curated.get("Source or origin") or source.get("source_origin", "Unknown"),
            "creators": curated.get("Who creates it") or ("qdg-metadata-notebook-1.0" if stage != "input" else "Unknown"),
            "access": curated.get("Who can access it") or "Unknown",
            "modifiers": curated.get("Who can modify it") or "Unknown",
            "curation_evidence": curated.get("Evidence or notes") or "Needs curator review",
            "filesystem_mtime_utc": datetime.fromtimestamp(path.stat().st_mtime, timezone.utc).isoformat(),
            "extraction_started_utc": started, "extracted_at_utc": now(), "status": status, "error": error,
            "technical": technical, "git_commit": source.get("commit"),
            "git_blob_sha1": source.get("git_blob_sha1")}

for path in sorted(INPUT.rglob("*")):
    if path.is_file() and not path.name.startswith("."):
        assets.append(asset(path))

# Preserve the executable source used in this run, independent of notebook output cells.
notebook_json = json.loads(NOTEBOOK.read_text(encoding="utf-8"))
code_source = "\n\n".join("".join(c["source"]) for c in notebook_json["cells"] if c["cell_type"] == "code")
code_path = RUN / "executed_notebook_source.py"
code_path.write_text(code_source, encoding="utf-8")
code_asset = asset(code_path, "run specification", "Code cells of metadata_provenance_atlas.ipynb")
config_asset = asset(RUN / "run_config.json", "run specification", "Configuration recorded by this execution")
annotations_asset = asset(RUN / "curator_annotations.json", "run specification", "Curator annotations loaded for this run")
assets.extend([code_asset, config_asset, annotations_asset])
# The manifests supply origin and version statements used in the metadata output.
for name in ["input_manifest.json", "repository_snapshot.json"]:
    manifest_asset = asset(PROJECT_ROOT / name, "run specification", "Recorded during package assembly")
    manifest_asset["creators"] = "Package assembly process"
    assets.append(manifest_asset)
show([{k: a[k] for k in ["name", "format", "size_bytes", "status"]} for a in assets])


[
  {
    "name": "queerdatagap_clean_frequency_matrix_LGBTQ_vocabulary.ipynb",
    "format": "Jupyter notebook",
    "size_bytes": 39809,
    "status": "ok"
  },
  {
    "name": "queerdatagraph_explorando_terminos_relaciones_SRT.ipynb",
    "format": "Jupyter notebook",
    "size_bytes": 3522440,
    "status": "ok"
  },
  {
    "name": "Glosario_TDSyG_WEB.pdf",
    "format": "PDF document",
    "size_bytes": 2317052,
    "status": "ok"
  },
  {
    "name": "Resentir_lo_queer_en_America_Latina.pdf",
    "format": "PDF document",
    "size_bytes": 17927136,
    "status": "ok"
  },
  {
    "name": "QueerDataGap_Lexico_Situado.xlsx",
    "format": "Excel workbook",
    "size_bytes": 498068,
    "status": "ok"
  },
  {
    "name": "lista_podcasts.xlsx",
    "format": "Excel workbook",
    "size_bytes": 57919,
    "status": "ok"
  },
  {
    "name": "fvfRMweIiWw.srt",
    "format": "SRT transcript",
    "size_bytes": 75442,
    "status": "ok"
  },
  {
    "name": "h-CkQVuco1Q.srt",
    "for

## 4 Create CSV derivatives and record process provenance

Each spreadsheet sheet becomes a CSV with `_source_row` pointing to the original Excel row. SRT derivatives contain the cue text and original timestamp windows. These are local research artifacts; Atlas receives structural metadata and lineage, not their full text. No claim is made that these derivatives are outputs of the downloaded GitHub analyses.

A process event records the actual source dataset, the executed notebook source and configuration, the output dataset, UTC start and end times, the responsible software agent and parameters. An optional curator identifier records responsibility when supplied. The eight lexicon sheets remain separate, preserving their different meanings and regional context.


In [5]:
def write_csv(path, headers, rows):
    with path.open("w", encoding="utf-8-sig", newline="") as stream:
        writer = csv.writer(stream)
        writer.writerow(headers)
        writer.writerows(rows)

def record_process(name, inputs, outputs, started, parameters):
    event = {"id": f"{PROJECT_ID}://run/{RUN_ID}/process/{len(activities) + 1}",
             "name": name, "started_at_utc": started, "ended_at_utc": now(),
             "inputs": [a["qualified_name"] for a in inputs],
             "outputs": [a["qualified_name"] for a in outputs],
             "agent": "qdg-metadata-notebook-1.0", "curator_id": CURATOR_ID,
             "parameters": parameters, "code_sha256": code_asset["sha256"],
             "status": "completed"}
    activities.append(event)

for original in list(assets):
    path = PROJECT_ROOT / original["relative_path"]
    if original["stage"] != "input" or original["status"] != "ok":
        continue
    started = now()
    if path.suffix.lower() == ".xlsx":
        for index, t in enumerate(xlsx_tables(path), 1):
            started = now()
            # Numbered names avoid collisions caused by accented or punctuation-heavy sheet names.
            destination = RUN / f"{path.stem}__sheet_{index}.csv"
            write_csv(destination, ["_source_row"] + t["columns"], [[rownum] + row for rownum, row in t["rows"]])
            derived = asset(destination, "derived", original["qualified_name"] + "#sheet=" + quote(t["sheet"]))
            assets.append(derived)
            record_process("Export spreadsheet sheet to CSV", [original, code_asset, config_asset], [derived], started,
                           {"sheet": t["sheet"], "header_row": t["header_row"], "blank_rows": "excluded",
                            "formulas": "preserved as expressions, not recalculated", "encoding": "utf-8-sig"})
    elif path.suffix.lower() == ".srt":
        cues, issues = parse_srt(path)
        destination = RUN / f"{path.stem}__cues.csv"
        fields = ["source_block", "cue_id", "start_seconds", "end_seconds", "text"]
        write_csv(destination, fields, [[cue[k] for k in fields] for cue in cues])
        derived = asset(destination, "derived", original["qualified_name"])
        assets.append(derived)
        record_process("Parse SRT cues to CSV", [original, code_asset, config_asset], [derived], started,
                       {"timestamp_unit": "seconds", "text": "preserved", "reported_issues": issues})

# Create a metadata artifact generated by this extraction run. It does not contain itself.
started = now()
metadata_path = RUN / "extracted_metadata.json"
dump(metadata_path, assets)
metadata_asset = asset(metadata_path, "metadata output", "Metadata extracted in run " + RUN_ID)
record_process("Extract and assemble metadata", list(assets), [metadata_asset],
               min(a["extraction_started_utc"] for a in assets), {"versions": versions, "profilers": "format-specific"})
assets.append(metadata_asset)
dump(RUN / "asset_inventory.json", assets)
dump(RUN / "process_events.json", activities)
print(f"Recorded {len(assets)} artifacts and {len(activities)} completed processes.")


Recorded 27 artifacts and 13 completed processes.


## 5 Export a PROV-JSON representation

The W3C PROV model distinguishes entities (artifacts), activities (operations) and agents (responsible software or people). `used` connects an operation to inputs; `wasGeneratedBy` connects outputs to the operation; `wasAssociatedWith` identifies the agent. Derivation edges connect data inputs to outputs, while code and configuration remain explicitly recorded as used inputs. This captures observed processing here, not an invented history of the supplied transcripts or books.


In [6]:
prov = {"prefix": {"prov": "http://www.w3.org/ns/prov#", "qdg": "https://example.org/queerdatagap/"},
        "entity": {}, "activity": {}, "agent": {}, "used": {}, "wasGeneratedBy": {},
        "wasAssociatedWith": {}, "wasDerivedFrom": {}}
entity_ids = {a["qualified_name"]: f"qdg:e{i}" for i, a in enumerate(assets, 1)}
for a in assets:
    prov["entity"][entity_ids[a["qualified_name"]]] = {
        "prov:label": a["name"], "qdg:qualifiedName": a["qualified_name"], "qdg:sha256": a["sha256"]}
prov["agent"]["qdg:software"] = {"prov:type": {"$": "prov:SoftwareAgent", "type": "prov:QUALIFIED_NAME"},
                                          "prov:label": "Metadata notebook 1.0"}
if CURATOR_ID != "unknown":
    prov["agent"]["qdg:curator"] = {"prov:label": CURATOR_ID}
for i, event in enumerate(activities, 1):
    aid = f"qdg:a{i}"
    prov["activity"][aid] = {"prov:label": event["name"], "prov:startTime": event["started_at_utc"],
                            "prov:endTime": event["ended_at_utc"], "qdg:parameters": json.dumps(event["parameters"])}
    prov["wasAssociatedWith"][f"qdg:assoc{i}"] = {"prov:activity": aid, "prov:agent": "qdg:software"}
    if CURATOR_ID != "unknown":
        prov["wasAssociatedWith"][f"qdg:curator{i}"] = {"prov:activity": aid, "prov:agent": "qdg:curator"}
    for j, name in enumerate(event["inputs"]):
        prov["used"][f"qdg:u{i}_{j}"] = {"prov:activity": aid, "prov:entity": entity_ids[name]}
    for k, name in enumerate(event["outputs"]):
        prov["wasGeneratedBy"][f"qdg:g{i}_{k}"] = {"prov:entity": entity_ids[name], "prov:activity": aid}
        for j, source in enumerate(event["inputs"]):
            if source in [code_asset["qualified_name"], config_asset["qualified_name"]]:
                continue
            prov["wasDerivedFrom"][f"qdg:d{i}_{k}_{j}"] = {
                "prov:generatedEntity": entity_ids[name], "prov:usedEntity": entity_ids[source], "prov:activity": aid}
dump(RUN / "provenance.prov.json", prov)
print("PROV entities:", len(prov["entity"]), "activities:", len(prov["activity"]))


PROV entities: 27 activities: 13


## 6 Map artifacts and processes to Apache Atlas

This notebook assumes **Apache Atlas**, using its v2 REST API. `qdg_artifact_v1` extends `DataSet`, so PDFs, transcripts, tables and code can participate in lineage. `qdg_process_v1` extends `Process` and uses its inherited `inputs` and `outputs` references. The software agent is represented by process attributes in Atlas and as a distinct agent in PROV-JSON. This is a project-specific mapping, not a native Atlas implementation of all PROV concepts.

Fields such as hashes and file format are directly queryable attributes. Format-specific nested metadata are serialized in `technicalMetadataJson`; promoting frequently queried fields to typed Atlas attributes is a possible next extension. Atlas server audit fields such as `createdBy` and `createTime` are not used to represent original research authorship. Recording access rights as metadata does not enforce permissions.

Temporary negative GUIDs link entities within one bulk payload. `qualifiedName` identifies existing datasets on subsequent imports. Run identifiers distinguish separate executions. The API payloads include metadata only, although names and embedded metadata can still require review before sharing.


In [7]:
ARTIFACT_TYPE = "qdg_artifact_v1"
PROCESS_TYPE = "qdg_process_v1"

def attribute(name, kind="string", optional=True):
    return {"name": name, "typeName": kind, "isOptional": optional, "cardinality": "SINGLE",
            "valuesMinCount": 0 if optional else 1, "valuesMaxCount": 1,
            "isUnique": False, "isIndexable": False}

artifact_fields = ["relativePath", "sha256", "fileFormat", "stage", "sourceOrigin", "creators",
                   "accessRights", "modificationRights", "curationEvidence", "filesystemMtimeUtc",
                   "extractedAtUtc", "extractionStatus", "extractionError", "technicalMetadataJson",
                   "gitCommit", "gitBlobSha1"]
process_fields = ["runId", "startedAtUtc", "endedAtUtc", "softwareAgent", "curatorId",
                  "parametersJson", "codeSha256", "executionStatus"]
typedefs = {"enumDefs": [], "structDefs": [], "classificationDefs": [], "relationshipDefs": [],
           "entityDefs": [
               {"name": ARTIFACT_TYPE, "superTypes": ["DataSet"], "typeVersion": "1.0",
                "description": "A versioned research artifact in the curation workflow",
                "attributeDefs": [attribute(x) for x in artifact_fields] + [attribute("sizeBytes", "long")]},
               {"name": PROCESS_TYPE, "superTypes": ["Process"], "typeVersion": "1.0",
                "description": "An observed execution with input and output lineage",
                "attributeDefs": [attribute(x) for x in process_fields]}]}
guids = {a["qualified_name"]: str(-i) for i, a in enumerate(assets, 1)}
entities = []
mapping = {"relative_path": "relativePath", "sha256": "sha256", "size_bytes": "sizeBytes",
           "format": "fileFormat", "stage": "stage", "source_origin": "sourceOrigin", "creators": "creators",
           "access": "accessRights", "modifiers": "modificationRights", "curation_evidence": "curationEvidence",
           "filesystem_mtime_utc": "filesystemMtimeUtc", "extracted_at_utc": "extractedAtUtc",
           "status": "extractionStatus", "error": "extractionError", "git_commit": "gitCommit", "git_blob_sha1": "gitBlobSha1"}
for a in assets:
    attrs = {target: a[source] for source, target in mapping.items() if a.get(source) is not None}
    attrs.update({"qualifiedName": a["qualified_name"], "name": a["name"],
                  "technicalMetadataJson": json.dumps(a["technical"], ensure_ascii=False)})
    entities.append({"typeName": ARTIFACT_TYPE, "guid": guids[a["qualified_name"]], "attributes": attrs})
for i, event in enumerate(activities, len(assets) + 1):
    attrs = {"qualifiedName": event["id"], "name": event["name"], "runId": RUN_ID,
             "startedAtUtc": event["started_at_utc"], "endedAtUtc": event["ended_at_utc"],
             "softwareAgent": event["agent"], "curatorId": CURATOR_ID,
             "parametersJson": json.dumps(event["parameters"], ensure_ascii=False),
             "codeSha256": event["code_sha256"], "executionStatus": event["status"]}
    for direction in ["inputs", "outputs"]:
        attrs[direction] = [{"guid": guids[name], "typeName": ARTIFACT_TYPE} for name in event[direction]]
    entities.append({"typeName": PROCESS_TYPE, "guid": str(-i), "attributes": attrs})
payload = {"entities": entities}
dump(RUN / "atlas_typedefs.json", typedefs)
dump(RUN / "atlas_entities.json", payload)
print("Atlas payload:", len(entities), "entities")


Atlas payload: 40 entities


## 7 Validate and review before import

These checks verify local references, required names, hashes and process times. They do not replace validation by the destination Atlas server. Review extraction errors, SRT issues and curator annotations. No live Atlas endpoint was provided for this package, so server compatibility and authentication must be tested in your deployment.

`asset_inventory.json` is the run-specific catalogue and `process_events.json` is the execution log. The accompanying Excel catalogue is an editable documentation snapshot; new results are written to JSON and CSV so reruns do not overwrite your notes. The metadata schema workbook describes the contract rather than acting as executable type configuration.


In [8]:
qnames = [e["attributes"]["qualifiedName"] for e in entities]
assert len(qnames) == len(set(qnames)), "Duplicate entity qualified names"
assert all(e["attributes"].get("name") for e in entities)
assert all(re.fullmatch(r"[0-9a-f]{64}", a["sha256"]) for a in assets)
for event in activities:
    assert set(event["inputs"] + event["outputs"]) <= set(guids)
    assert not set(event["inputs"]) & set(event["outputs"])
    assert datetime.fromisoformat(event["started_at_utc"]) <= datetime.fromisoformat(event["ended_at_utc"])
for entry in manifest + repo_snapshot:
    assert sha256(PROJECT_ROOT / entry["relative_path"]) == entry["sha256"], "Input changed during run"
review = {"artifact_count": len(assets), "process_count": len(activities),
          "extraction_errors": [{"path": a["relative_path"], "error": a["error"]} for a in assets if a["status"] == "error"],
          "srt_issues": [{"path": a["relative_path"], "issues": a["technical"]["timestamp_issues"]}
                         for a in assets if a["technical"].get("timestamp_issues")],
          "unknown_creators": sum(a["creators"].lower() == "unknown" for a in assets),
          "local_reference_checks": "passed", "atlas_server_validation": "not performed"}
dump(RUN / "validation_report.json", review)
write_csv(RUN / "data_catalogue_snapshot.csv",
          ["qualified_name", "name", "source_origin", "format", "creators", "access", "modifiers", "relative_path", "sha256", "status"],
          [[a[k] for k in ["qualified_name", "name", "source_origin", "format", "creators", "access", "modifiers", "relative_path", "sha256", "status"]]
          for a in assets])
show([review])


[
  {
    "artifact_count": 27,
    "process_count": 13,
    "extraction_errors": [],
    "srt_issues": [],
    "unknown_creators": 9,
    "local_reference_checks": "passed",
    "atlas_server_validation": "not performed"
  }
]


## 8 Optional Atlas submission

Leave `SUBMIT_TO_ATLAS = False` for local use. After reviewing the generated files, set the flag and provide `ATLAS_URL` (server origin, for example `https://atlas.example.org`) plus `ATLAS_USER` and `ATLAS_PASSWORD` as environment variables. This example supports HTTP Basic authentication over HTTPS; Kerberos or other institutional authentication requires adapting the client. Credentials are not written to the provenance files. TLS verification remains enabled.

The client first reads each type definition. It creates missing types only and refuses incompatible existing definitions. It does not silently modify shared types. Bulk entity submission then creates or updates entities by their unique attributes. Configure the server base path if your deployment uses a reverse proxy. Keep the input documents local; only the two metadata JSON payloads are submitted.


In [9]:
SUBMIT_TO_ATLAS = False

def submit_to_atlas():
    import base64, urllib.request, urllib.error
    base = os.environ.get("ATLAS_URL", "").rstrip("/")
    user = os.environ.get("ATLAS_USER", "")
    password = os.environ.get("ATLAS_PASSWORD", "")
    if not base.startswith("https://") or not user or not password:
        raise ValueError("Set HTTPS ATLAS_URL, ATLAS_USER and ATLAS_PASSWORD before submission.")
    auth = base64.b64encode(f"{user}:{password}".encode()).decode()

    def request(method, path, body=None):
        data = json.dumps(body).encode() if body is not None else None
        req = urllib.request.Request(base + "/api/atlas/v2" + path, data=data, method=method,
                                     headers={"Authorization": "Basic " + auth,
                                              "Content-Type": "application/json", "Accept": "application/json"})
        with urllib.request.urlopen(req, timeout=60) as response:
            return json.load(response)

    missing = []
    for desired in typedefs["entityDefs"]:
        try:
            current = request("GET", "/types/entitydef/name/" + desired["name"])
        except urllib.error.HTTPError as exc:
            if exc.code == 404:
                missing.append(desired)
                continue
            raise
        actual = {a["name"]: a["typeName"] for a in current.get("attributeDefs", [])}
        if not set(desired["superTypes"]) <= set(current.get("superTypes", [])) or any(
            actual.get(a["name"]) != a["typeName"] for a in desired["attributeDefs"]
        ):
            raise ValueError(f"Atlas type mismatch: {desired['name']}. Review a versioned type migration.")
    if missing:
        request("POST", "/types/typedefs", {"entityDefs": missing})
    response = request("POST", "/entity/bulk", payload)
    dump(RUN / "atlas_submission_response.json", response)
    print("Atlas submission completed; response saved locally.")

if SUBMIT_TO_ATLAS:
    submit_to_atlas()
else:
    print("Local export complete. Atlas submission is disabled.")


Local export complete. Atlas submission is disabled.


## Interpretation and next curation steps

Confirm transcript source URLs against the podcast register, identify the transcription method and model version if available, document collection dates and consent or reuse conditions where applicable, and record the responsible creator and editing roles. The register's analyst column is evidence of assigned analysis, not necessarily of transcription authorship.

If later analyses generate a frequency matrix, filtered vocabulary, inverted index or graph, call `record_process` at the point of execution with the actual input/output entities, code version and parameters. Preserve term-selection decisions, stop-word lists, stemming settings and manual revisions. Do not attach a GitHub commit to a historical output without evidence of that execution. Reference documents may inform interpretation without being computational inputs to every step.

### Sources

- [Project notebooks at the recorded commit](https://github.com/gevargas/queerdatagap/tree/94fba639e34048cfe6ba57bddf8cd2ef5c606e4f/notebooks)
- [Apache Atlas type system and lineage concepts](https://atlas.apache.org/2.0.0/TypeSystem.html)
- [Apache Atlas v2 type-definition API](https://atlas.apache.org/api/v2/resource_TypesREST.html)
- [Apache Atlas v2 entity API](https://atlas.apache.org/api/v2/resource_EntityREST.html)
- [W3C PROV overview](https://www.w3.org/TR/prov-overview/)
- Supplied XLSX header rows, SRT structures and PDF embedded metadata are the local evidence for this package. Their content has not been independently verified.
